# 04b Ablation — Per-Category & Per-Fold AUROC (100% Labels)

This notebook loads `../results/results.parquet` produced by `src/evaluation/aggregate.py` and produces:

- Mean validation AUROC ± std per category (across 3 folds)
- Per-fold AUROC breakdown per category
- PNG figures saved to `../results/figures/` at ≥150 DPI

> **Scope note:** Label-ratio ablation (10%/50%/100%) was dropped. Variance now comes from 3-fold CV.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

RESULTS_PATH = Path("../results/results.parquet")
FIG_DIR = Path("../results/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

assert RESULTS_PATH.exists(), f"Missing required parquet: {RESULTS_PATH}"

In [ ]:
df = pd.read_parquet(RESULTS_PATH)

# Schema validation
required_cols = ["category", "fold", "val_auroc", "val_f1", "run_id", "run_name"]
missing_cols = [c for c in required_cols if c not in df.columns]
assert not missing_cols, f"Parquet missing required columns: {missing_cols}"

df = df.copy()
df["fold"] = df["fold"].astype(int)

# Verify coverage: 6 categories × 3 folds
EXPECTED_CATEGORIES = {"bottle", "capsule", "carpet", "hazelnut", "leather", "pill"}
EXPECTED_FOLDS = {1, 2, 3}

present_categories = set(df["category"].dropna().unique().tolist())
present_folds = set(df["fold"].dropna().astype(int).unique().tolist())

missing_categories = EXPECTED_CATEGORIES - present_categories
missing_folds = EXPECTED_FOLDS - present_folds

assert not missing_categories, f"Missing categories: {sorted(missing_categories)}"
assert not missing_folds, f"Missing folds: {sorted(missing_folds)}"

null_val = df[df["val_auroc"].isna()]
if not null_val.empty:
    offenders = null_val["run_id"].astype(str).tolist()
    raise ValueError(f"val_auroc contains null values for runs: {offenders}")

df = df.sort_values(["category", "fold", "run_id"]).reset_index(drop=True)
print(f"Rows loaded: {len(df)}")
print(f"Categories: {sorted(present_categories)}")
print(f"Folds: {sorted(present_folds)}")
df[["category", "fold", "run_name", "val_auroc", "val_f1", "run_id"]].head(18)

In [ ]:
# --- Figure 1: Mean Validation AUROC ± std per Category ---
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({"savefig.dpi": 200, "figure.dpi": 140})

cat_agg = (
    df.groupby("category", as_index=False)
      .agg(
          mean_val_auroc=("val_auroc", "mean"),
          std_val_auroc=("val_auroc", "std"),
      )
      .sort_values("mean_val_auroc", ascending=False)
)
cat_agg["std_val_auroc"] = cat_agg["std_val_auroc"].fillna(0.0)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(
    cat_agg["category"],
    cat_agg["mean_val_auroc"],
    yerr=cat_agg["std_val_auroc"],
    capsize=5,
    color=sns.color_palette("viridis", len(cat_agg)),
    edgecolor="white",
    linewidth=0.8,
    error_kw={"linewidth": 1.5},
)
ax.set_title("Mean Validation AUROC per Category (100% Labels, 3-Fold CV)")
ax.set_xlabel("Defect Category")
ax.set_ylabel("Validation AUROC (0 to 1)")
ax.set_ylim(0.8, 1.02)
ax.axhline(cat_agg["mean_val_auroc"].mean(), color="red", linestyle="--",
           linewidth=1.5, label=f"Overall mean = {cat_agg['mean_val_auroc'].mean():.4f}")
ax.legend(frameon=True)
fig.tight_layout()

fig1 = FIG_DIR / "ablation_mean_val_auroc_per_category.png"
fig.savefig(fig1, dpi=200)
plt.show()
cat_agg

In [ ]:
# --- Figure 2: Per-Fold AUROC Breakdown per Category ---
fig, ax = plt.subplots(figsize=(12, 6))

for category, grp in df.groupby("category"):
    ax.errorbar(
        grp["fold"],
        grp["val_auroc"],
        marker="o",
        linewidth=2,
        capsize=3,
        label=category,
    )

ax.set_title("Per-Fold Validation AUROC by Category (100% Labels)")
ax.set_xlabel("Fold")
ax.set_ylabel("Validation AUROC (0 to 1)")
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(["Fold 1", "Fold 2", "Fold 3"])
ax.set_ylim(0.8, 1.02)
ax.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()

fig2 = FIG_DIR / "ablation_per_fold_val_auroc_by_category.png"
fig.savefig(fig2, dpi=200)
plt.show()

print("Saved figures:")
print(f"  - {fig1}")
print(f"  - {fig2}")

# Per-fold summary table
df[["category", "fold", "run_name", "val_auroc", "val_f1"]]